In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
class AgeGenderCNN(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super(AgeGenderCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)
        )

        # Flatten sonrası boyutu otomatik hesapla
        dummy_input = torch.zeros(1, 1, *input_size)
        dummy_output = self.features(dummy_input)
        self.flattened_size = dummy_output.view(1, -1).shape[1]

        # Cinsiyet tahmini başlığı
        self.gender_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

        # Yaş tahmini başlığı
        self.age_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        gender_out = torch.sigmoid(self.gender_head(x))
        age_out = self.age_head(x)
        return gender_out, age_out


In [3]:
class AgeGenderDataset(Dataset):
    def __init__(self, X, y_gender, y_age):
        self.X = X.astype(np.float32) / 255.0
        self.y_gender = y_gender.astype(np.float32).reshape(-1, 1)
        self.y_age = y_age.astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)
        return torch.tensor(x), torch.tensor(self.y_gender[idx]), torch.tensor(self.y_age[idx])

In [4]:
def train_model(model, dataloader, criterion_g, criterion_a, optimizer, device):
    model.train()
    total_loss = 0
    for x, y_g, y_a in tqdm(dataloader):
        x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
        optimizer.zero_grad()
        pred_g, pred_a = model(x)
        loss_g = criterion_g(pred_g, y_g)
        loss_a = criterion_a(pred_a, y_a)
        loss = loss_g + loss_a
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [5]:
def evaluate_model(model, dataloader, criterion_g, criterion_a, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
            pred_g, pred_a = model(x)
            loss_g = criterion_g(pred_g, y_g)
            loss_a = criterion_a(pred_a, y_a)
            loss = loss_g + loss_a
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [6]:
def test_model(model, dataloader, device):
    model.eval()
    all_preds_g = []
    all_preds_a = []
    all_true_g = []
    all_true_a = []

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x = x.to(device)
            pred_g, pred_a = model(x)
            all_preds_g += pred_g.cpu().numpy().flatten().tolist()
            all_preds_a += pred_a.cpu().numpy().flatten().tolist()
            all_true_g += y_g.numpy().flatten().tolist()
            all_true_a += y_a.numpy().flatten().tolist()

    pred_g_bin = [1 if p > 0.5 else 0 for p in all_preds_g]

    print("\n=== [TEST SONUÇLARI] ===")
    print("Cinsiyet - Accuracy :", accuracy_score(all_true_g, pred_g_bin))
    print("Cinsiyet - Precision:", precision_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - Recall   :", recall_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - F1-score :", f1_score(all_true_g, pred_g_bin, zero_division=0))

    print("Yaş - MAE           :", mean_absolute_error(all_true_a, all_preds_a))
    print("Yaş - RMSE          :", root_mean_squared_error(all_true_a, all_preds_a))

In [7]:
X_train = np.load("X_train_utkface.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_utkface.npy")
y_gen_train = np.load("y_gender_train_utkface.npy")

X_test = np.load("X_test_utkface.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_utkface.npy")
y_gen_test = np.load("y_gender_test_utkface.npy")
valid_mask = (y_gen_train == 0) | (y_gen_train == 1)

X_train = X_train[valid_mask]
y_gen_train = y_gen_train[valid_mask]
y_age_train = y_age_train[valid_mask]

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeGenderCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_utk2.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")
    

100%|██████████| 536/536 [00:12<00:00, 41.38it/s]


Epoch 1: Train Loss = 410.9066 | Val Loss = 339.2070
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:11<00:00, 46.47it/s]


Epoch 2: Train Loss = 320.7078 | Val Loss = 287.6132
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:11<00:00, 45.47it/s]


...

Epoch 28: Train Loss = 64.0233 | Val Loss = 301.8878


100%|██████████| 536/536 [00:27<00:00, 19.84it/s]


Epoch 29: Train Loss = 65.4245 | Val Loss = 364.1782


100%|██████████| 536/536 [00:27<00:00, 19.19it/s]


Epoch 30: Train Loss = 65.2637 | Val Loss = 294.2472


In [8]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN().to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.743385132297354
Cinsiyet - Precision: 0.724823870700373
Cinsiyet - Recall   : 0.7581274382314694
Cinsiyet - F1-score : 0.7411016949152542
Yaş - MAE           : 11.766564293638192
Yaş - RMSE          : 15.926017378770048


In [9]:
X_train = np.load("X_train_all_imdbwiki.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_all_imdbwiki.npy")
y_gen_train = np.load("y_gender_train_all_imdbwiki.npy")

X_test = np.load("X_test_all_imdbwiki.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_all_imdbwiki.npy")
y_gen_test = np.load("y_gender_test_all_imdbwiki.npy")

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderCNN(input_size=(64, 64)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_imdbwiki2.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")

100%|██████████| 11459/11459 [02:34<00:00, 74.17it/s] 


Epoch 1: Train Loss = 191.7352 | Val Loss = 182.3854
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [01:58<00:00, 96.98it/s] 


Epoch 2: Train Loss = 170.4905 | Val Loss = 155.7238
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [01:58<00:00, 96.98it/s] 


...

Epoch 28: Train Loss = 90.2987 | Val Loss = 147.9620


100%|██████████| 11459/11459 [02:29<00:00, 76.89it/s]


Epoch 29: Train Loss = 88.5646 | Val Loss = 146.6487


100%|██████████| 11459/11459 [02:28<00:00, 77.10it/s]


Epoch 30: Train Loss = 86.8439 | Val Loss = 148.1672


In [10]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN(input_size=(64, 64)).to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.7530215707258643
Cinsiyet - Precision: 0.7566100260875697
Cinsiyet - Recall   : 0.8717303005686434
Cinsiyet - F1-score : 0.8101007813384667
Yaş - MAE           : 9.009845156686985
Yaş - RMSE          : 11.86415251597464
